# Deepfake Voice Detection — Data Preparation
### Step 1 of 4: Install libraries, convert MP3 to WAV, update metadata, and split into chunks
---

In [1]:
import subprocess
for pkg in ['librosa', 'soundfile']:
    subprocess.run(['pip', 'install', pkg], capture_output=True)
    print(f'{pkg} installed successfully!')
print('All libraries installed!')

librosa installed successfully!
soundfile installed successfully!
All libraries installed!


## Step 1 — Import Libraries

In [2]:
import os
import csv
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import subprocess
from datetime import datetime

print('Libraries imported successfully!')

Libraries imported successfully!


## Step 2 — Set Paths and Convert MP3 to WAV

In [3]:
# AUTO PATH DETECTION — works on any machine
BASE_DIR    = os.path.dirname(os.path.abspath('__file__'))
REAL_FOLDER = os.path.join(BASE_DIR, 'dataset', 'real')
FAKE_FOLDER = os.path.join(BASE_DIR, 'dataset', 'fake')
CSV_PATH    = os.path.join(BASE_DIR, 'metadata.csv')

print(f'Project directory : {BASE_DIR}')
print(f'Real folder       : {REAL_FOLDER}')
print(f'Fake folder       : {FAKE_FOLDER}')
print()

if not os.path.exists(REAL_FOLDER):
    print(f'ERROR: Real folder not found: {REAL_FOLDER}')
elif not os.path.exists(FAKE_FOLDER):
    print(f'ERROR: Fake folder not found: {FAKE_FOLDER}')
else:
    mp3_files = [f for f in os.listdir(FAKE_FOLDER) if f.endswith('.mp3')]
    if mp3_files:
        print(f'Converting {len(mp3_files)} MP3 files to WAV...\n')
        success, failed = 0, 0
        for mp3_file in mp3_files:
            mp3_path = os.path.join(FAKE_FOLDER, mp3_file)
            wav_name = mp3_file.replace('.mp3', '.wav')
            wav_path = os.path.join(FAKE_FOLDER, wav_name)
            if os.path.exists(wav_path):
                print(f'Skipped (WAV exists): {wav_name}')
                continue
            result = subprocess.run(
                ['ffmpeg', '-i', mp3_path, '-ar', '16000',
                 '-ac', '1', '-sample_fmt', 's16', '-y', wav_path],
                capture_output=True, text=True
            )
            if os.path.exists(wav_path):
                os.remove(mp3_path)
                print(f'Converted: {mp3_file} -> {wav_name}')
                success += 1
            else:
                print(f'FAILED: {mp3_file}  |  {result.stderr[-150:]}')
                failed += 1
        print(f'\nConverted: {success}  |  Failed: {failed}\n')
    else:
        print('No MP3 files found — all fake files already WAV!\n')

    real_files = sorted([f for f in os.listdir(REAL_FOLDER) if f.endswith('.wav')])
    fake_files = sorted([f for f in os.listdir(FAKE_FOLDER) if f.endswith('.wav')])
    print(f'Real files : {len(real_files)}')
    print(f'Fake files : {len(fake_files)}')
    print(f'Total      : {len(real_files) + len(fake_files)}')
    if len(real_files) == len(fake_files):
        print('\nDataset balanced! Ready for training.')
    else:
        print(f'\nDataset unbalanced — {abs(len(real_files) - len(fake_files))} files difference.')

Project directory : d:\Hamza\NIAI\NIAI_Project\deepfake-voice-detection
Real folder       : d:\Hamza\NIAI\NIAI_Project\deepfake-voice-detection\dataset\real
Fake folder       : d:\Hamza\NIAI\NIAI_Project\deepfake-voice-detection\dataset\fake

No MP3 files found — all fake files already WAV!

Real files : 151
Fake files : 151
Total      : 302

Dataset balanced! Ready for training.


## Step 3 — Update Dataset CSV
Save all file records into metadata.csv with their labels (real/fake).

In [4]:
rows = []
for fname in sorted(real_files):
    rows.append({'filename': fname, 'label': 'real', 'source': 'youtube',
                 'language_mix': 'urdu+english',
                 'date_added': datetime.today().strftime('%Y-%m-%d')})
for fname in sorted(fake_files):
    rows.append({'filename': fname, 'label': 'fake', 'source': 'elevenlabs',
                 'language_mix': 'urdu+english',
                 'date_added': datetime.today().strftime('%Y-%m-%d')})

with open(CSV_PATH, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

df = pd.read_csv(CSV_PATH)
print(f'metadata.csv updated successfully!')
print(f'Total records : {len(df)}')
print(f'Real records  : {len(df[df["label"] == "real"])}')
print(f'Fake records  : {len(df[df["label"] == "fake"])}')
print()
print(df.head(3).to_string(index=False))

metadata.csv updated successfully!
Total records : 302
Real records  : 151
Fake records  : 151

    filename label  source language_mix date_added
real_001.wav  real youtube urdu+english 2026-05-28
real_002.wav  real youtube urdu+english 2026-05-28
real_003.wav  real youtube urdu+english 2026-05-28


## Step 4 — Split All Audio Clips into 3-Second Chunks
Each audio clip is divided into multiple 3-second chunks.
- Each chunk is exactly 3 seconds (48000 samples at 16kHz)
- Chunks shorter than 3 seconds are discarded
- Original files are kept safe — chunks are saved to new folders

In [5]:
TARGET_SR     = 16000
TARGET_LENGTH = 3 * TARGET_SR  # 48000 samples per chunk

REAL_CHUNKS_FOLDER = os.path.join(BASE_DIR, 'dataset', 'real_chunks')
FAKE_CHUNKS_FOLDER = os.path.join(BASE_DIR, 'dataset', 'fake_chunks')
os.makedirs(REAL_CHUNKS_FOLDER, exist_ok=True)
os.makedirs(FAKE_CHUNKS_FOLDER, exist_ok=True)

def split_into_chunks(file_path, output_folder, base_name):
    '''Split one audio file into 3-second chunks. Returns number of chunks created.'''
    try:
        audio, sr = librosa.load(file_path, sr=TARGET_SR, mono=True)
        n_chunks  = len(audio) // TARGET_LENGTH
        saved     = 0
        for i in range(n_chunks):
            start = i * TARGET_LENGTH
            chunk = audio[start:start + TARGET_LENGTH]
            sf.write(os.path.join(output_folder, f'{base_name}_chunk{i+1:02d}.wav'),
                     chunk, TARGET_SR)
            saved += 1
        return saved
    except Exception as e:
        print(f'Error: {file_path} -> {e}')
        return 0

print('Splitting real files into 3-second chunks...')
real_total = sum(
    split_into_chunks(os.path.join(REAL_FOLDER, f), REAL_CHUNKS_FOLDER,
                      os.path.splitext(f)[0])
    for f in sorted(real_files)
)
print(f'Real chunks created : {real_total}')

print('\nSplitting fake files into 3-second chunks...')
fake_total = sum(
    split_into_chunks(os.path.join(FAKE_FOLDER, f), FAKE_CHUNKS_FOLDER,
                      os.path.splitext(f)[0])
    for f in sorted(fake_files)
)
print(f'Fake chunks created : {fake_total}')

real_chunk_files = sorted([f for f in os.listdir(REAL_CHUNKS_FOLDER) if f.endswith('.wav')])
fake_chunk_files = sorted([f for f in os.listdir(FAKE_CHUNKS_FOLDER) if f.endswith('.wav')])

print(f'\n{"="*50}')
print(f'Original real files : {len(real_files)}  |  Original fake files : {len(fake_files)}')
print(f'Real chunks         : {len(real_chunk_files)}  |  Fake chunks         : {len(fake_chunk_files)}')
print(f'Total chunks        : {len(real_chunk_files) + len(fake_chunk_files)}')
print(f'{"="*50}')
print('Original files are safe in dataset/real/ and dataset/fake/')
print('Chunks saved to dataset/real_chunks/ and dataset/fake_chunks/')
print('Ready for MFCC feature extraction — run notebook 02_model_training.ipynb next!')

Splitting real files into 3-second chunks...
Real chunks created : 536

Splitting fake files into 3-second chunks...
Fake chunks created : 327

Original real files : 151  |  Original fake files : 151
Real chunks         : 536  |  Fake chunks         : 327
Total chunks        : 863
Original files are safe in dataset/real/ and dataset/fake/
Chunks saved to dataset/real_chunks/ and dataset/fake_chunks/
Ready for MFCC feature extraction — run notebook 02_model_training.ipynb next!
